In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {DEVICE}")
print(f"torch  = {torch.__version__}")

device = cpu
torch  = 2.2.2


In [2]:
class SingleHeadAttention(nn.Module):
    """Minimal single-head self-attention (Q = K = V = x)."""
    def __init__(self, d_model: int):
        super().__init__()
        self.d_k = d_model
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, D = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)

        if mask is not None:
            if mask.dim() == 2:                       # [T, T] -> [B, T, T]
                mask = mask.unsqueeze(0).expand(B, -1, -1)
            scores = scores.masked_fill(mask, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        return self.W_o(out), attn


print("SingleHeadAttention defined")

SingleHeadAttention defined


In [3]:
class MultiHeadAttention(nn.Module):
    """Multi-head self/cross-attention.
       Accepts [B, T, D]; returns [B, T, D] outputs and [B, H, Tq, Tk] weights."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        B, T_q, _ = query.shape
        T_k = key.shape[1]
        H, d_k = self.n_heads, self.d_k

        Q = self.W_q(query).view(B, T_q, H, d_k).transpose(1, 2)   # [B,H,Tq,dk]
        K = self.W_k(key).view(B, T_k, H, d_k).transpose(1, 2)     # [B,H,Tk,dk]
        V = self.W_v(value).view(B, T_k, H, d_k).transpose(1, 2)   # [B,H,Tk,dk]

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        attn = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, V)                                # [B,H,Tq,dk]
        out = out.transpose(1, 2).contiguous().view(B, T_q, self.d_model)
        return self.W_o(out), attn


print("MultiHeadAttention defined")

MultiHeadAttention defined


In [4]:
def reference_attention(Q, K, V, mask=None):
    """PyTorch built-in reference (no learned projections, no dropout)."""
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    attn = F.softmax(scores, dim=-1)
    out = torch.matmul(attn, V)
    return out, attn


def banner(title):
    print("=" * 60)
    print(title)
    print("=" * 60)


print("reference_attention + banner defined")

reference_attention + banner defined


In [5]:
def check_shapes():
    B, T, D, H = 2, 7, 16, 4
    m = MultiHeadAttention(D, H).to(DEVICE).eval()
    x = torch.randn(B, T, D, device=DEVICE)
    out, w = m(x, x, x)
    print("[CHECK 1] shapes")
    print(f"  output shape  : {tuple(out.shape)}  expected ({B},{T},{D})")
    print(f"  weights shape : {tuple(w.shape)}  expected ({B},{H},{T},{T})")
    assert out.shape == (B, T, D)
    assert w.shape == (B, H, T, T)
    print("  RESULT: PASS\n")


check_shapes()

[CHECK 1] shapes
  output shape  : (2, 7, 16)  expected (2,7,16)
  weights shape : (2, 4, 7, 7)  expected (2,4,7,7)
  RESULT: PASS



In [6]:
def check_softmax_sums():
    B, T, D, H = 2, 7, 16, 4
    m = MultiHeadAttention(D, H).to(DEVICE).eval()
    x = torch.randn(B, T, D, device=DEVICE)
    _, w = m(x, x, x)
    max_dev = (w.sum(dim=-1) - 1.0).abs().max().item()
    print("[CHECK 2] softmax row sums")
    print(f"  max |row_sum - 1| : {max_dev:.2e}   tolerance 1e-6")
    assert max_dev < 1e-6
    print("  RESULT: PASS\n")


check_softmax_sums()

[CHECK 2] softmax row sums
  max |row_sum - 1| : 1.19e-07   tolerance 1e-6
  RESULT: PASS



In [7]:
def check_causal_invariance():
    """Changing future tokens must not change earlier outputs (causal mask)."""
    B, T, D = 2, 10, 64
    x = torch.randn(B, T, D, device=DEVICE)
    model = SingleHeadAttention(D).to(DEVICE).eval()
    # True = masked. Upper triangle (future) = True.
    mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=DEVICE), diagonal=1)

    out_causal, _ = model(x, mask=mask)

    x_mod = x.clone()
    x_mod[:, 5:, :] = torch.randn_like(x_mod[:, 5:, :])
    out_causal_mod, _ = model(x_mod, mask=mask)

    first_5_unchanged = torch.allclose(out_causal[:, :5], out_causal_mod[:, :5])
    last_5_changed    = not torch.allclose(out_causal[:, 5:], out_causal_mod[:, 5:])

    print("[CHECK 3] causal invariance")
    print(f"  first 5 outputs unchanged : {first_5_unchanged}")
    print(f"  last 5 outputs changed    : {last_5_changed}")
    assert first_5_unchanged, "Past outputs changed despite causal mask."
    assert last_5_changed, "Future outputs unchanged — causal mask not active."
    print("  RESULT: PASS\n")


check_causal_invariance()

[CHECK 3] causal invariance
  first 5 outputs unchanged : True
  last 5 outputs changed    : True
  RESULT: PASS



In [8]:
def check_gradients():
    B, T, D, H = 2, 7, 16, 4
    m = MultiHeadAttention(D, H).to(DEVICE).train()
    x = torch.randn(B, T, D, device=DEVICE, requires_grad=True)
    out, _ = m(x, x, x)
    out.sum().backward()
    for name, p in m.named_parameters():
        assert p.grad is not None, f"missing grad for {name}"
        assert torch.isfinite(p.grad).all(), f"non-finite grad in {name}"
    print("[CHECK 4] finite gradients")
    print("  all parameter grads finite : YES")
    print("  RESULT: PASS\n")


check_gradients()

[CHECK 4] finite gradients
  all parameter grads finite : YES
  RESULT: PASS



In [9]:
def check_masked_keys():
    B, T, D, H = 1, 5, 16, 4
    m = MultiHeadAttention(D, H).to(DEVICE).eval()
    x = torch.randn(B, T, D, device=DEVICE)

    key_pad = torch.zeros(1, 1, 1, T, dtype=torch.bool, device=DEVICE)
    key_pad[..., 3:] = True                      # pad last 2 keys
    _, w = m(x, x, x, mask=key_pad)

    masked_max = w[..., 3:].abs().max().item()
    valid_dev  = (w[..., :3].sum(dim=-1) - 1.0).abs().max().item()

    print("[CHECK 5] masked-key probabilities")
    print(f"  max prob on masked keys  : {masked_max:.2e}   expected 0.0")
    print(f"  max |valid_key_sum - 1|  : {valid_dev:.2e}   tolerance 1e-6")
    assert masked_max < 1e-9
    assert valid_dev < 1e-6
    print("  RESULT: PASS\n")


check_masked_keys()

[CHECK 5] masked-key probabilities
  max prob on masked keys  : 0.00e+00   expected 0.0
  max |valid_key_sum - 1|  : 5.96e-08   tolerance 1e-6
  RESULT: PASS



In [10]:
def check_reference_match():
    B, T, D, H = 2, 6, 16, 4
    m = MultiHeadAttention(D, H).to(DEVICE).eval()
    x = torch.randn(B, T, D, device=DEVICE)

    Q = m.W_q(x).view(B, T, H, m.d_k).transpose(1, 2)
    K = m.W_k(x).view(B, T, H, m.d_k).transpose(1, 2)
    V = m.W_v(x).view(B, T, H, m.d_k).transpose(1, 2)

    with torch.no_grad():
        out_ref, w_ref = reference_attention(Q, K, V)
        out_ref = out_ref.transpose(1, 2).contiguous().view(B, T, D)
        out_ref = m.W_o(out_ref)
        out_mine, w_mine = m(x, x, x)

    w_diff   = (w_mine - w_ref).abs().max().item()
    out_diff = (out_mine - out_ref).abs().max().item()

    print("[CHECK 6] reference match")
    print(f"  max |Δ weights|  : {w_diff:.2e}   tolerance 1e-6")
    print(f"  max |Δ output|   : {out_diff:.2e}   tolerance 1e-5")
    assert w_diff   < 1e-6
    assert out_diff < 1e-5
    print("  RESULT: PASS\n")


check_reference_match()

[CHECK 6] reference match
  max |Δ weights|  : 0.00e+00   tolerance 1e-6
  max |Δ output|   : 0.00e+00   tolerance 1e-5
  RESULT: PASS



In [11]:
def check_cross_attention_lengths():
    """Cross-attention with T_q != T_k; masked encoder keys must get 0 prob."""
    B, D, H = 2, 16, 4
    T_q, T_k = 5, 9
    m = MultiHeadAttention(D, H).to(DEVICE).eval()

    q = torch.randn(B, T_q, D, device=DEVICE)
    k = torch.randn(B, T_k, D, device=DEVICE)
    v = torch.randn(B, T_k, D, device=DEVICE)

    enc_mask = torch.zeros(1, 1, 1, T_k, dtype=torch.bool, device=DEVICE)
    enc_mask[..., -2:] = True                      # last 2 keys are PAD

    out, w = m(q, k, v, mask=enc_mask)

    print("[CROSS-ATTN 1] shape with T_q != T_k")
    print(f"  T_q={T_q}, T_k={T_k}")
    print(f"  output shape   : {tuple(out.shape)}  expected ({B},{T_q},{D})")
    print(f"  weights shape  : {tuple(w.shape)}  expected ({B},{H},{T_q},{T_k})")
    assert out.shape == (B, T_q, D)
    assert w.shape   == (B, H, T_q, T_k)
    print("  RESULT: PASS\n")

    masked_probs = w[..., -2:].abs().max().item()
    valid_sums   = w[..., :-2].sum(dim=-1)
    max_dev      = (valid_sums - 1.0).abs().max().item()

    print("[CROSS-ATTN 2] mask correctness with T_q != T_k")
    print(f"  max prob on masked encoder keys : {masked_probs:.2e}   expected 0.0")
    print(f"  max |valid_key_sum - 1|        : {max_dev:.2e}   tolerance 1e-6")
    assert masked_probs < 1e-9
    assert max_dev < 1e-6
    print("  RESULT: PASS\n")


check_cross_attention_lengths()

[CROSS-ATTN 1] shape with T_q != T_k
  T_q=5, T_k=9
  output shape   : (2, 5, 16)  expected (2,5,16)
  weights shape  : (2, 4, 5, 9)  expected (2,4,5,9)
  RESULT: PASS

[CROSS-ATTN 2] mask correctness with T_q != T_k
  max prob on masked encoder keys : 0.00e+00   expected 0.0
  max |valid_key_sum - 1|        : 1.19e-07   tolerance 1e-6
  RESULT: PASS



In [12]:
def check_cross_attention_gradient():
    """Gradient from decoder query must reach encoder value."""
    B, D, H = 2, 16, 4
    T_q, T_k = 5, 9
    m = MultiHeadAttention(D, H).to(DEVICE).eval()

    q  = torch.randn(B, T_q, D, device=DEVICE)
    k  = torch.randn(B, T_k, D, device=DEVICE)
    v2 = torch.randn(B, T_k, D, device=DEVICE).requires_grad_(True)

    enc_mask = torch.zeros(1, 1, 1, T_k, dtype=torch.bool, device=DEVICE)
    enc_mask[..., -2:] = True

    out2, _ = m(q, k, v2, mask=enc_mask)
    out2.sum().backward()

    assert v2.grad is not None and torch.isfinite(v2.grad).all()
    grad_norm = v2.grad.norm().item()

    print("[CROSS-ATTN 3] gradient flows to encoder value")
    print(f"  ||∂out/∂v|| = {grad_norm:.4f}  (must be > 0)")
    assert grad_norm > 0
    print("  RESULT: PASS\n")


check_cross_attention_gradient()

[CROSS-ATTN 3] gradient flows to encoder value
  ||∂out/∂v|| = 3.7794  (must be > 0)
  RESULT: PASS



In [13]:
def check_cross_attention_asymmetric_padding():
    """Per-sample encoder padding of different lengths."""
    B, D, H = 3, 16, 4
    T_q, T_k = 4, 7
    m = MultiHeadAttention(D, H).to(DEVICE).eval()

    q = torch.randn(B, T_q, D, device=DEVICE)
    k = torch.randn(B, T_k, D, device=DEVICE)
    v = torch.randn(B, T_k, D, device=DEVICE)

    pad_lengths = [4, 1, 0]                        # per-sample encoder pad
    enc_mask = torch.zeros(B, 1, 1, T_k, dtype=torch.bool, device=DEVICE)
    for i, p in enumerate(pad_lengths):
        if p > 0:
            enc_mask[i, ..., T_k - p:] = True

    out, w = m(q, k, v, mask=enc_mask)

    print("[CROSS-ATTN 4] per-sample encoder padding")
    for i, p in enumerate(pad_lengths):
        if p > 0:
            masked_prob = w[i, :, :, T_k - p:].abs().max().item()
            valid_sum   = w[i, :, :, :T_k - p].sum(dim=-1)
        else:
            masked_prob = 0.0
            valid_sum   = w[i, :, :, :].sum(dim=-1)
        max_dev = (valid_sum - 1.0).abs().max().item()
        print(f"  sample {i}: pad={p}  masked_max={masked_prob:.2e}  valid_dev={max_dev:.2e}")
        assert masked_prob < 1e-9
        assert max_dev < 1e-6
    print("  RESULT: PASS\n")


check_cross_attention_asymmetric_padding()

[CROSS-ATTN 4] per-sample encoder padding
  sample 0: pad=4  masked_max=0.00e+00  valid_dev=1.19e-07
  sample 1: pad=1  masked_max=0.00e+00  valid_dev=1.19e-07
  sample 2: pad=0  masked_max=0.00e+00  valid_dev=1.19e-07
  RESULT: PASS



In [14]:
banner("WEEK 4 — FULL VERIFICATION SUMMARY")
print("Six self-attention checks:")
print("  [1] shapes                     PASS")
print("  [2] softmax row sums           PASS")
print("  [3] causal invariance          PASS")
print("  [4] finite gradients           PASS")
print("  [5] masked-key probabilities   PASS")
print("  [6] reference match            PASS")
print()
print("Four cross-attention checks:")
print("  [C1] T_q != T_k shapes         PASS")
print("  [C2] mask correctness          PASS")
print("  [C3] gradient to encoder value PASS")
print("  [C4] per-sample padding        PASS")
print()
banner("ALL 10 CHECKS PASSED")

WEEK 4 — FULL VERIFICATION SUMMARY
Six self-attention checks:
  [1] shapes                     PASS
  [2] softmax row sums           PASS
  [3] causal invariance          PASS
  [4] finite gradients           PASS
  [5] masked-key probabilities   PASS
  [6] reference match            PASS

Four cross-attention checks:
  [C1] T_q != T_k shapes         PASS
  [C2] mask correctness          PASS
  [C3] gradient to encoder value PASS
  [C4] per-sample padding        PASS

ALL 10 CHECKS PASSED


In [15]:
print("Measured deviations (float32):")
print("-" * 60)

# Check 2
B, T, D, H = 2, 7, 16, 4
m = MultiHeadAttention(D, H).to(DEVICE).eval()
x = torch.randn(B, T, D, device=DEVICE)
_, w = m(x, x, x)
print(f"[2] softmax row-sum deviation        : {(w.sum(-1)-1).abs().max().item():.2e}")

# Check 3
B, T, D = 2, 10, 64
x = torch.randn(B, T, D, device=DEVICE)
m3 = SingleHeadAttention(D).to(DEVICE).eval()
mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=DEVICE), diagonal=1)
o1, _ = m3(x, mask=mask)
x2 = x.clone(); x2[:, 5:, :] = torch.randn_like(x2[:, 5:, :])
o2, _ = m3(x2, mask=mask)
print(f"[3] causal invariance Δ earlier out  : {(o1[:, :5]-o2[:, :5]).abs().max().item():.2e}")

# Check 5
B, T, D, H = 1, 5, 16, 4
m5 = MultiHeadAttention(D, H).to(DEVICE).eval()
x = torch.randn(B, T, D, device=DEVICE)
kp = torch.zeros(1,1,1,T, dtype=torch.bool, device=DEVICE); kp[..., 3:] = True
_, w = m5(x, x, x, mask=kp)
print(f"[5] masked-key max probability       : {w[...,3:].abs().max().item():.2e}")
print(f"[5] valid-key row-sum deviation      : {(w[...,:3].sum(-1)-1).abs().max().item():.2e}")

# Check 6
B, T, D, H = 2, 6, 16, 4
m6 = MultiHeadAttention(D, H).to(DEVICE).eval()
x = torch.randn(B, T, D, device=DEVICE)
Q = m6.W_q(x).view(B,T,H,m6.d_k).transpose(1,2)
K = m6.W_k(x).view(B,T,H,m6.d_k).transpose(1,2)
V = m6.W_v(x).view(B,T,H,m6.d_k).transpose(1,2)
with torch.no_grad():
    out_ref, w_ref = reference_attention(Q, K, V)
    out_ref = m6.W_o(out_ref.transpose(1,2).contiguous().view(B,T,D))
    out_mine, w_mine = m6(x, x, x)
print(f"[6] reference weight max |Δ|         : {(w_mine-w_ref).abs().max().item():.2e}")
print(f"[6] reference output max |Δ|         : {(out_mine-out_ref).abs().max().item():.2e}")

# Cross-attention 2
B, D, H = 2, 16, 4; T_q, T_k = 5, 9
mc = MultiHeadAttention(D, H).to(DEVICE).eval()
q = torch.randn(B,T_q,D, device=DEVICE)
k = torch.randn(B,T_k,D, device=DEVICE)
v = torch.randn(B,T_k,D, device=DEVICE)
em = torch.zeros(1,1,1,T_k, dtype=torch.bool, device=DEVICE); em[..., -2:] = True
_, w = mc(q,k,v, mask=em)
print(f"[C2] masked encoder-key max prob     : {w[...,-2:].abs().max().item():.2e}")
print(f"[C2] valid-key row-sum deviation     : {(w[...,:-2].sum(-1)-1).abs().max().item():.2e}")

# Cross-attention 3
v2 = v.clone().requires_grad_(True)
out2, _ = mc(q, k, v2, mask=em)
out2.sum().backward()
print(f"[C3] ||∂out/∂v||                     : {v2.grad.norm().item():.4f}")

# Cross-attention 4
B, D, H = 3, 16, 4; T_q, T_k = 4, 7
ma = MultiHeadAttention(D, H).to(DEVICE).eval()
q = torch.randn(B,T_q,D, device=DEVICE)
k = torch.randn(B,T_k,D, device=DEVICE)
v = torch.randn(B,T_k,D, device=DEVICE)
pads = [4, 1, 0]
em = torch.zeros(B,1,1,T_k, dtype=torch.bool, device=DEVICE)
for i,p in enumerate(pads):
    if p > 0: em[i, ..., T_k-p:] = True
_, w = ma(q,k,v, mask=em)
for i,p in enumerate(pads):
    if p > 0:
        mp = w[i,:,:,T_k-p:].abs().max().item()
        vd = (w[i,:,:,:T_k-p].sum(-1)-1).abs().max().item()
    else:
        mp, vd = 0.0, (w[i].sum(-1)-1).abs().max().item()
    print(f"[C4] sample {i} pad={p}  masked_max={mp:.2e}  valid_dev={vd:.2e}")

Measured deviations (float32):
------------------------------------------------------------
[2] softmax row-sum deviation        : 1.19e-07
[3] causal invariance Δ earlier out  : 0.00e+00
[5] masked-key max probability       : 0.00e+00
[5] valid-key row-sum deviation      : 1.19e-07
[6] reference weight max |Δ|         : 0.00e+00
[6] reference output max |Δ|         : 0.00e+00
[C2] masked encoder-key max prob     : 0.00e+00
[C2] valid-key row-sum deviation     : 1.19e-07
[C3] ||∂out/∂v||                     : 3.9062
[C4] sample 0 pad=4  masked_max=0.00e+00  valid_dev=1.19e-07
[C4] sample 1 pad=1  masked_max=0.00e+00  valid_dev=1.19e-07
[C4] sample 2 pad=0  masked_max=0.00e+00  valid_dev=1.79e-07
